# Regresión de cuantiles: piso y techo (P10/P50/P90)

## Qué responde este notebook

Los modelos elegidos en [`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb)
dan un solo número por receptor y semana: el valor esperado. Ese número no dice qué tan incierta es
la predicción; un rango, con un piso y un techo, sí. Aquí se prueba si se puede estimar ese rango con regresión de cuantiles: P10 (un piso,
el valor que el receptor supera 9 de cada 10 veces), P50 (la mediana) y P90 (un techo).

Un cuantil se evalúa con su propia métrica, la pérdida *pinball*, que es la que premia predecir ese
cuantil (Gneiting, 2011), y con la cobertura: qué fracción de los resultados reales cae entre P10 y
P90 (lo esperado es 80%). Se usa una configuración fija, sin búsqueda de hiperparámetros.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
train_mask = wr["season"].between(2016, 2021)
val_mask = wr["season"].between(2022, 2023)
test_mask = wr["season"].between(2024, 2025)
print(f"entrenamiento: {train_mask.sum()} filas (2016-2021) · validacion: {val_mask.sum()} (2022-2023) · prueba: {test_mask.sum()} (2024-2025)")

entrenamiento: 13795 filas (2016-2021) · validacion: 4863 (2022-2023) · prueba: 4952 (2024-2025)


## 1. Un caso concreto antes que cualquier métrica

Ja'Marr Chase, el receptor que se revisó a fondo en
[`2.6_eda_jugador_destacado.ipynb`](2.6_eda_jugador_destacado.ipynb), en sus últimas 3 semanas de 2025: un modelo de cuantiles 0.1, 0.5 y 0.9 para
`receiving_yards`, entrenado con 2016-2021, contra lo que de verdad pasó.

In [2]:
CUANTILES = [0.1, 0.5, 0.9]

def entrenar_cuantiles(target, features_arbol, train_mask):
    modelos = {}
    for q in CUANTILES:
        m = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=200, random_state=0)
        m.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
        modelos[q] = m
    return modelos

modelos_yardas = entrenar_cuantiles("receiving_yards", features_arbol, train_mask)

chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
chase_test = wr[test_mask & (wr["player_id"] == chase_id)].sort_values(["season", "week"]).tail(3)
for _, fila in chase_test.iterrows():
    X_fila = fila[features_arbol].to_frame().T
    p10 = max(0, modelos_yardas[0.1].predict(X_fila)[0])
    p50 = max(0, modelos_yardas[0.5].predict(X_fila)[0])
    p90 = max(0, modelos_yardas[0.9].predict(X_fila)[0])
    print(f"season={int(fila.season)} week={int(fila.week)}: real={fila.receiving_yards:.0f} yardas -- "
          f"P10={p10:.0f} · P50={p50:.0f} · P90={p90:.0f}")

season=2025 week=16: real=109 yardas -- P10=33 · P50=74 · P90=148
season=2025 week=17: real=60 yardas -- P10=34 · P50=80 · P90=137
season=2025 week=18: real=96 yardas -- P10=34 · P50=70 · P90=138


**Lectura.** El resultado real cae dentro de la banda P10–P90 en las tres semanas (109, 60 y 96
yardas, con bandas de 33–148, 34–137 y 34–138), y dos de las tres veces queda por encima de P50. Con
tres casos eso no dice nada general; la sección 3 lo mide sobre toda la prueba.

## 2. Qué algoritmo usar para los cuantiles

XGBoost tiene un objetivo de cuantiles propio (`reg:quantileerror`, los tres cuantiles en un solo
modelo); `HistGradientBoostingRegressor` necesita un modelo por cuantil. Se comparan en
**validación** (2022-2023), con yardas como caso de prueba, XGBoost con su configuración por defecto
y HistGradientBoosting con 200 iteraciones. Se miden pinball, porcentaje de filas con cuantiles
cruzados (P10 > P50 o P50 > P90, algo que no debería pasar) y cobertura. El que gane se usa para los
tres resultados.

In [3]:
def pinball_loss(y_true, y_pred, q):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

y_val_yardas = wr.loc[val_mask, "receiving_yards"].to_numpy()

modelo_xgb_nativo = XGBRegressor(objective="reg:quantileerror", quantile_alpha=CUANTILES, random_state=0, n_jobs=-1)
modelo_xgb_nativo.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, "receiving_yards"])
pred_xgb = np.clip(modelo_xgb_nativo.predict(wr.loc[val_mask, features_arbol]), 0, None)

pred_hgb = np.column_stack([
    np.clip(modelos_yardas[q].predict(wr.loc[val_mask, features_arbol]), 0, None) for q in CUANTILES
])

comparacion = pd.DataFrame({
    "pinball_xgb_nativo": [pinball_loss(y_val_yardas, pred_xgb[:, i], q) for i, q in enumerate(CUANTILES)],
    "pinball_hgb": [pinball_loss(y_val_yardas, pred_hgb[:, i], q) for i, q in enumerate(CUANTILES)],
}, index=CUANTILES)
print("pinball en validacion, receiving_yards (menor es mejor):")
print(comparacion.round(3).to_string())

viola_xgb = ((pred_xgb[:, 0] > pred_xgb[:, 1]) | (pred_xgb[:, 1] > pred_xgb[:, 2])).mean()
viola_hgb = ((pred_hgb[:, 0] > pred_hgb[:, 1]) | (pred_hgb[:, 1] > pred_hgb[:, 2])).mean()
cobertura_xgb = ((y_val_yardas >= pred_xgb[:, 0]) & (y_val_yardas <= pred_xgb[:, 2])).mean()
cobertura_hgb = ((y_val_yardas >= pred_hgb[:, 0]) & (y_val_yardas <= pred_hgb[:, 2])).mean()
print(f"\n% filas con cuantiles cruzados (P10>P50 o P50>P90): xgb_nativo={viola_xgb:.1%}, hgb={viola_hgb:.1%}")
print(f"cobertura empirica P10-P90 (deberia ser ~80%): xgb_nativo={cobertura_xgb:.1%}, hgb={cobertura_hgb:.1%}")

pinball en validacion, receiving_yards (menor es mejor):
     pinball_xgb_nativo  pinball_hgb
0.1               3.481        3.187
0.5              10.770       10.412
0.9               6.365        6.238

% filas con cuantiles cruzados (P10>P50 o P50>P90): xgb_nativo=3.5%, hgb=0.0%
cobertura empirica P10-P90 (deberia ser ~80%): xgb_nativo=62.7%, hgb=82.4%


**Lectura.** En validación, HistGradientBoosting gana en los tres cuantiles de pinball (3.187 contra
3.481 en P10, 10.412 contra 10.770 en P50 y 6.238 contra 6.365 en P90), no cruza cuantiles (0.0%
contra 3.5%) y cubre 82.4% de los resultados reales entre P10 y P90, contra 62.7% del objetivo
nativo de XGBoost. Se usa HistGradientBoosting, con un modelo por cuantil. Como no tiene un formato equivalente
al de XGBoost, sus modelos se guardan con joblib.

## 3. Los tres resultados, evaluados en prueba

Con el algoritmo elegido en validación, se reportan pinball, monotonía y cobertura en la prueba
2024-2025.

In [4]:
TARGETS = ["receptions", "receiving_yards", "receiving_tds"]

modelos_por_target = {}
predicciones_test = {}
for target in TARGETS:
    modelos_por_target[target] = entrenar_cuantiles(target, features_arbol, train_mask)
    predicciones_test[target] = {
        q: np.clip(modelos_por_target[target][q].predict(wr.loc[test_mask, features_arbol]), 0, None)
        for q in CUANTILES
    }

y_test = {t: wr.loc[test_mask, t].to_numpy() for t in TARGETS}

tabla_pinball = pd.DataFrame({
    target: {q: pinball_loss(y_test[target], predicciones_test[target][q], q) for q in CUANTILES}
    for target in TARGETS
}).T
print("pinball loss por target y cuantil:")
print(tabla_pinball.round(4).to_string())

pinball loss por target y cuantil:
                    0.1     0.5     0.9
receptions       0.2379  0.6680  0.3708
receiving_yards  3.0045  9.7978  5.8743
receiving_tds    0.0203  0.1017  0.0969


In [5]:
filas_monotonia = []
filas_cobertura = []
for target in TARGETS:
    p10, p50, p90 = predicciones_test[target][0.1], predicciones_test[target][0.5], predicciones_test[target][0.9]
    viola = ((p10 > p50) | (p50 > p90))
    filas_monotonia.append({"target": target, "filas_violadas": viola.sum(), "total": len(p10), "%": 100*viola.mean()})

    dentro = (y_test[target] >= p10) & (y_test[target] <= p90)
    debajo_p50 = y_test[target] <= p50
    filas_cobertura.append({
        "target": target,
        "cobertura_P10_P90": dentro.mean(),
        "P(real<=P50)": debajo_p50.mean(),
        "%_P10_es_cero": (p10 < 0.01).mean(),
        "%_P50_es_cero": (p50 < 0.01).mean(),
    })

print("monotonia (P10<=P50<=P90):")
print(pd.DataFrame(filas_monotonia).to_string(index=False))
print("\ncobertura empirica (P10-P90 deberia rondar 80%, P(real<=P50) deberia rondar 50%):")
print(pd.DataFrame(filas_cobertura).round(3).to_string(index=False))

monotonia (P10<=P50<=P90):
         target  filas_violadas  total        %
     receptions               6   4952 0.121163
receiving_yards               1   4952 0.020194
  receiving_tds               0   4952 0.000000

cobertura empirica (P10-P90 deberia rondar 80%, P(real<=P50) deberia rondar 50%):
         target  cobertura_P10_P90  P(real<=P50)  %_P10_es_cero  %_P50_es_cero
     receptions              0.862         0.545          0.808          0.034
receiving_yards              0.828         0.540          0.536          0.005
  receiving_tds              0.888         0.823          1.000          1.000


**Lectura.** En recepciones y yardas el rango funciona como se espera: casi ninguna fila con
cuantiles cruzados (6 de 4,952 en recepciones y 1 en yardas) y cobertura de 86.2% y 82.8%, algo por
encima del 80%. La mediana queda un poco alta: 54.5% y 54.0% de los resultados reales caen por debajo
de P50, en lugar de 50%. En recepciones, P10 vale prácticamente 0 en 80.8% de las filas: el piso de
un receptor típico es no atrapar nada. En touchdowns el rango casi no
informa: P10 y P50 valen 0 en todas las filas de prueba. Es coherente con un resultado que es 0 en
82.3% de los partidos de prueba: una mediana de 0 quiere decir que el modelo estima, para cada
receptor y semana, que no anotar es al menos tan probable como anotar. La cobertura sube a 88.8%. Toda la información está en P90: cuántos touchdowns podría anotar en una
semana muy buena.

## 4. ¿Y si se usara la mediana (P50) como predicción puntual?

El P50 minimiza el error absoluto. Se compara contra los modelos elegidos en `4.4` con la métrica
principal de cada resultado, en la prueba: es el ejemplo práctico de por qué el MAE no decide en
este proyecto (ver [`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb)).

In [6]:
CONFIG_ELEGIDA = {
    "receptions": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="count:poisson", max_depth=3, n_estimators=400, learning_rate=0.03),
}

filas = {}
for target, params in CONFIG_ELEGIDA.items():
    modelo_punto = XGBRegressor(**params, random_state=0, n_jobs=-1)
    modelo_punto.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
    pred_punto = np.clip(modelo_punto.predict(wr.loc[test_mask, features_arbol]), 0, None)
    media = wr.loc[train_mask, target].mean()
    for nombre, pred in (("modelo elegido", pred_punto), ("P50 (mediana)", predicciones_test[target][0.5])):
        m = modeling.metricas_target(target, y_test[target], pred, media_entrenamiento=media)
        filas[(target, nombre)] = {k: m.get(k) for k in ("mae", "rmse", "r2_oos", "deviance_poisson", "d2_oos", "sesgo")}
pd.DataFrame(filas).T.astype(float).round(4)

mae     rmse  r2_oos  deviance_poisson  d2_oos   sesgo
receptions      modelo elegido   1.3697   1.8355  0.4527            1.3980  0.4493  0.0577
                P50 (mediana)    1.3360   1.8602  0.4379            1.5607  0.3853 -0.2252
receiving_yards modelo elegido  20.5522  27.9349  0.3744               NaN     NaN  1.2251
                P50 (mediana)   19.5955  28.7443  0.3377               NaN     NaN -5.6560
receiving_tds   modelo elegido   0.3020   0.4454  0.0914            0.6221  0.1399 -0.0044
                P50 (mediana)    0.2034   0.5094 -0.1887            5.2868 -6.3094 -0.2034

**Lectura.** Es el patrón de `4.1` con modelos reales. La mediana gana en MAE en los tres resultados
(1.3360 contra 1.3697 en recepciones, 19.60 contra 20.55 en yardas y 0.2034 contra 0.3020 en
touchdowns), pero pierde en RMSE y en R² fuera de muestra en los tres, y predice de menos: sesgo de
−0.23 recepciones y −5.66 yardas, porque en distribuciones con cola a la derecha, como las de recepciones y
yardas (ver [`2.1_eda_targets.ipynb`](2.1_eda_targets.ipynb)), la mediana queda por debajo de la media. En touchdowns es el caso extremo: la mediana es 0 para todos, su MAE es el mejor y
su D² es −6.31. Como predicción puntual se queda el valor esperado; el P50 solo tiene sentido como
parte del rango.

## 5. Guardar los modelos de cuantil

Se guardan solo los del algoritmo elegido en la sección 2, como artefactos exploratorios en
`weekly/wr/models/exploratorio/`: el flujo semanal no los usa.

In [7]:
import json, os
import joblib

os.makedirs("../models/exploratorio", exist_ok=True)
metadata_cuantiles = {}
for target in TARGETS:
    for q in CUANTILES:
        etiqueta = f"p{int(q*100)}"
        ruta = f"../models/exploratorio/{target}_{etiqueta}.joblib"
        joblib.dump(modelos_por_target[target][q], ruta)

    fila_cobertura = next(f for f in filas_cobertura if f["target"] == target)
    fila_monotonia = next(f for f in filas_monotonia if f["target"] == target)
    metadata_cuantiles[target] = {
        "target": target,
        "modelo": "HistGradientBoostingRegressor(loss='quantile')",
        "cuantiles": CUANTILES,
        "hiperparametros": {"max_iter": 200, "random_state": 0},
        "formato_guardado": "joblib (no JSON nativo -- HistGradientBoostingRegressor no tiene un equivalente; ver seccion 2 de 4.5_regresion_cuantiles.ipynb)",
        "features_arbol_en_orden": features_arbol,
        "entrenado_con": "entrenamiento 2016-2021, mismas particiones que la etapa 4",
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "pinball_loss_test_2024_2025": {str(q): round(pinball_loss(y_test[target], predicciones_test[target][q], q), 4) for q in CUANTILES},
        "cobertura_empirica_P10_P90_test_2024_2025": round(fila_cobertura["cobertura_P10_P90"], 4),
        "limitaciones_conocidas": [
            "P10 y P50 salen en 0 para el 100% de las filas en receiving_tds (target 82% cero) -- el rango solo aporta senal real en P90 para ese target",
            f"filas con cuantiles cruzados (P10 > P50 o P50 > P90) en test 2024-2025: {fila_monotonia['filas_violadas']} de {fila_monotonia['total']}",
        ],
    }
    with open(f"../models/exploratorio/{target}_cuantiles_metadata.json", "w") as f:
        json.dump(metadata_cuantiles[target], f, indent=2, ensure_ascii=False)

print("guardados:", sorted(f for f in os.listdir("../models/exploratorio") if "p10" in f or "p50" in f or "p90" in f or "cuantiles" in f))

guardados: ['receiving_tds_cuantiles_metadata.json', 'receiving_tds_p10.joblib', 'receiving_tds_p50.joblib', 'receiving_tds_p90.joblib', 'receiving_yards_cuantiles_metadata.json', 'receiving_yards_p10.joblib', 'receiving_yards_p50.joblib', 'receiving_yards_p90.joblib', 'receptions_cuantiles_metadata.json', 'receptions_p10.joblib', 'receptions_p50.joblib', 'receptions_p90.joblib']


## Cierre

- La regresión de cuantiles con HistGradientBoosting da un rango útil para recepciones y yardas
  (cobertura de 83% a 86%, casi sin cuantiles cruzados). Se eligió en validación sobre el objetivo
  nativo de cuantiles de XGBoost.
- En touchdowns solo P90 aporta información.
- El P50 no reemplaza al valor esperado: gana en MAE, pierde en RMSE y R² y predice de menos.
- Los modelos quedan en `weekly/wr/models/exploratorio/`; el flujo semanal no los usa.

Anterior: [`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb) · Siguiente:
[`4.6_ensamble_simple.ipynb`](4.6_ensamble_simple.ipynb)